# Лабораторная 4. Модель данных и витрины

После третьей работы у нас есть чистые таблицы. Казалось бы, работай.

Понедельник. Маркетинг спрашивает выручку по категориям. Аналитик пишет запрос
на пять соединений и получает число.

Вторник. Финансы спрашивают то же самое. Второй аналитик пишет свой запрос на
пять соединений и получает другое число: он посчитал выручку по платежам, а не
по позициям заказа, и не исключил отменённые заказы.

Среда. Совещание, на котором выясняется, что «выручка» в компании – это три
разных числа, и никто не может сказать, какое правильное. Все три запроса при
этом написаны корректно.

Проблема не в SQL. Проблема в том, что у компании нет общего определения
выручки, заказа и товара – каждый собирает его заново из очищенных таблиц.
Сегодня мы это определение зафиксируем.

## Перед началом: новые слова

| Термин | Простыми словами |
|---|---|
| **Хранилище данных – Data Warehouse, DWH** | Общие для компании таблицы, собранные под вопросы бизнеса, а не под работу приложения. |
| **Таблица фактов – fact table** | События бизнеса с числами, которые складывают: продажи, платежи, отгрузки. |
| **Измерение – dimension** | Справочник, задающий разрез: клиент, товар, продавец, дата. |
| **Схема «звезда» – star schema** | Факт в центре, справочники вокруг, соединение на один шаг. |
| **Витрина – data mart** | Готовая таблица под конкретный вопрос, собранная из модели. |
| **Суррогатный ключ – surrogate key** | Искусственный идентификатор строки справочника, не несущий делового смысла. |
| **Озеро данных – Data Lake** | Дешёвое хранилище файлов любого вида, схема применяется при чтении. |

## Подготовка

Ячейка подключает хранилище и читает слой CLEAN, собранный в третьей работе.

In [ ]:
# Готовим окружение: в VS Code и Jupyter берём открытую папку курса,
# в Colab материалы скачиваются один раз. Всё остальное – в src/notebook_env.py.
from pathlib import Path
import subprocess, sys

ROOT = next((c for s in (Path.cwd().resolve(), Path("/content/data-engineering-ai-course"))
             for c in (s, *s.parents) if (c / ".course-root").exists()), None)
if ROOT is None:                                   # чистая сессия Colab
    url = ''
    if not url:
        raise RuntimeError("Откройте папку курса в VS Code/Jupyter или склонируйте "
                           "репозиторий в /content/data-engineering-ai-course")
    subprocess.run(["git", "clone", "--depth", "1", url,
                    "/content/data-engineering-ai-course"], check=True)
    ROOT = Path("/content/data-engineering-ai-course")
sys.path[:0] = [str(ROOT), str(ROOT / "src")]
from notebook_env import setup_course
setup_course(ROOT)


In [ ]:
import duckdb
import pandas as pd
import paths
from lab_utils import require_artifact
from stages import ensure_stage

pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)

# Для независимого запуска в Colab воспроизводим результат предыдущей ЛР 3.
# Текущие CORE и MART студент по-прежнему строит сам ниже.
ensure_stage("clean")
for t in ["clean_orders", "clean_items", "clean_payments", "clean_reviews",
          "clean_customers", "clean_products", "clean_sellers"]:
    require_artifact(paths.CLEAN / f"{t}.parquet", "ЛР 3")

con = duckdb.connect(str(paths.WAREHOUSE))
con.execute("CREATE SCHEMA IF NOT EXISTS core")
con.execute("CREATE SCHEMA IF NOT EXISTS mart")

In [ ]:
from course_visuals import show_external_visual, show_visual

show_visual("star")

Факт хранит событие, измерения дают ему контекст. Набор измерений всегда
выбирается под те вопросы, на которые модель должна отвечать.

In [ ]:
for t in ["clean_orders", "clean_items", "clean_payments", "clean_reviews",
          "clean_customers", "clean_products", "clean_sellers"]:
    con.execute(f"""
        CREATE OR REPLACE TABLE {t} AS
        SELECT * FROM read_parquet('{(paths.CLEAN / (t + ".parquet")).as_posix()}')
    """)

con.execute("""
    SELECT 'clean_orders' AS таблица, COUNT(*) AS строк FROM clean_orders
    UNION ALL SELECT 'clean_items', COUNT(*) FROM clean_items
    UNION ALL SELECT 'clean_payments', COUNT(*) FROM clean_payments
    UNION ALL SELECT 'clean_reviews', COUNT(*) FROM clean_reviews
    UNION ALL SELECT 'clean_customers', COUNT(*) FROM clean_customers
    UNION ALL SELECT 'clean_products', COUNT(*) FROM clean_products
    UNION ALL SELECT 'clean_sellers', COUNT(*) FROM clean_sellers
""").df()

## Зачем отдельный слой, если таблицы уже чистые

Чистые таблицы отвечают на вопрос «что нам прислали». Они не отвечают на
вопрос «что такое выручка в этой компании». Пока определение живёт в голове
у каждого аналитика отдельно, спорить о цифрах можно бесконечно, и обычно
так и происходит.

Нужен слой, где определения записаны один раз и лежат на виду. Его называют
**хранилищем данных (Data Warehouse, DWH)**: централизованный набор таблиц,
собранный под вопросы бизнеса и под историю, а не под работу приложения.

Самое ценное слово в этом определении – «централизованный». Клиент из системы
продаж и клиент из биллинга должны стать одним клиентом, иначе любой отчёт
будет верен только для того, кто его написал.

Слои – это не бюрократия, а разделение ответственности:

| Слой | Что там лежит | Кто отвечает | Можно ли менять задним числом |
|------|---------------|--------------|-------------------------------|
| **RAW** | как приехало от источника | инженер | нет, слой неизменяемый |
| **STAGING / CLEAN** | типы приведены, мусор в карантине | инженер | пересобирается из RAW |
| **CORE** | бизнес-модель: факты и измерения, единые определения | инженер + аналитик | пересобирается из CLEAN |
| **MART** | витрины под конкретные вопросы и дашборды | аналитик | пересобирается из CORE |

Правило: **данные текут только вниз**. Витрина не читает RAW напрямую,
CORE не читает чужие витрины. Иначе через полгода у вас клубок, в котором
изменение одной колонки ломает три дашборда неизвестно где.

### DWH, Data Lake и Lakehouse

| | **DWH** | **Data Lake** | **Lakehouse** |
|---|---|---|---|
| Что хранит | таблицы со схемой | файлы любого вида | файлы + слой таблиц над ними |
| Схема | при записи (schema-on-write) | при чтении (schema-on-read) | при записи, но поверх файлов |
| Сильная сторона | быстрые надёжные отчёты | дёшево хранить всё, включая картинки и логи | одно хранилище и для BI, и для ML |
| Слабая сторона | дорого, только табличное | легко превращается в «болото данных» | сложнее в эксплуатации |
| Примеры | Snowflake, BigQuery, ClickHouse | S3/HDFS + Parquet | Delta Lake, Apache Iceberg |

В курсе мы используем файлы Parquet и DuckDB как лёгкую имитацию
аналитического хранилища. Это ещё не lakehouse: папка с Parquet не знает
о снимках данных, одновременной записи из двух процессов и изменении схемы.
Недостающая часть – слой метаданных:

```
  объектное хранилище
  + колоночные файлы
  + метаданные таблицы и транзакции
  ─────────────────────────────────
  = таблица lakehouse
```

Эти метаданные и добавляют Apache Iceberg и Delta Lake. Как выглядит таблица,
у которой они есть:

In [ ]:
show_external_visual(
    "iceberg_metadata",
    caption="Пунктир делит таблицу Iceberg на слой данных и слой метаданных.")

Внизу – те самые файлы с данными, обычный Parquet, ничем не отличающийся от
наших. Всё, что выше пунктира, – это и есть разница между папкой с файлами
и таблицей: `s0` и `s1` – снимки состояния, каждый помнит свой набор файлов,
поэтому можно прочитать таблицу такой, какой она была вчера, и дописать в неё
данные, не мешая чужому запросу. Остальные подробности этой схемы сейчас
не нужны.

## Что случилось и в каком разрезе

Размерное моделирование (Кимбалл) предлагает делить все таблицы на два вида.

> **Факт (fact)** – таблица событий бизнеса с числовыми мерами: продажа,
> платёж, клик, отгрузка. Строк много, они постоянно добавляются.
>
> **Измерение (dimension)** – таблица-справочник, отвечающая на вопрос
> «в каком разрезе смотрим»: клиент, товар, продавец, дата, регион.
> Строк мало, они меняются редко.

Простой тест: если в колонку хочется поставить `SUM()` – это мера (факт).
Если по ней хочется сделать `GROUP BY` – это атрибут измерения.

```
                     dim_date
                         │
       dim_customer ─── fact_order_items ─── dim_product
                         │           │
                    dim_seller       └── меры: price, freight_value
```

Это и есть **звезда (star schema)**: факт в центре, измерения вокруг, соединение
на один JOIN. Если измерение разложить на несколько нормализованных таблиц
(товар → подкатегория → категория), получится **снежинка (snowflake)** –
меньше дублирования, но больше JOIN и сложнее запросы. В аналитике обычно
выбирают звезду: место дешевле, чем время аналитика.

### Суррогатный ключ

> **Суррогатный ключ** – искусственный идентификатор строки измерения
> (обычно целое число), не имеющий бизнес-смысла.

Зачем он нужен: натуральный ключ может измениться (клиент сменил телефон,
товар переехал в другую категорию), может быть длинным (32-символьный hex)
и может приехать из разных систем в разном формате. Суррогатный ключ ещё
и позволяет хранить историю изменений измерения (SCD Type 2): у одного клиента
несколько строк с разными периодами действия.

Мы сделаем суррогатный ключ только для даты (`date_key` вида `20180317`),
а для остальных измерений оставим натуральные идентификаторы – они у нас
стабильные. Это нормальное инженерное упрощение, но знать про альтернативу нужно.

## Сначала зерно факта, потом запрос

> **Зерно факта (grain)** – что именно описывает одна строка таблицы фактов.

Для наших данных возможны как минимум три варианта:

| Зерно | Одна строка = | Чем хорош | Чем плох |
|-------|---------------|-----------|----------|
| позиция заказа | товар в заказе | можно резать по товару и продавцу | метрики уровня заказа нельзя суммировать |
| заказ | заказ целиком | удобно считать заказы и доставку | теряется товарный разрез |
| платёж | платёжная операция | видна структура оплат | не про товары вообще |

Правильного ответа «вообще» не существует – есть правильный ответ **под вопросы,
на которые должна отвечать модель**. Нам нужны и товарный разрез, и метрики
доставки, поэтому мы возьмём зерно «позиция заказа» и положим рядом атрибуты
заказа – но пометим их как **атрибуты уровня заказа, которые нельзя суммировать**.

Это важнейшая ловушка размерного моделирования:

```
  fact_order_items (зерно = позиция)
  ├── price           ← мера уровня позиции: SUM можно
  ├── freight_value   ← мера уровня позиции: SUM можно
  ├── delivery_days   ← атрибут уровня ЗАКАЗА: AVG по позициям ≠ AVG по заказам!
  └── is_late         ← атрибут уровня ЗАКАЗА: COUNT по позициям завышает
```

Если в витрине нужно «сколько заказов опоздало» – считать надо
`COUNT(DISTINCT order_id)` по строкам с `is_late`, а не `COUNT(*)`.

In [ ]:
# Демонстрация цены ошибки: заказ из пяти позиций опоздал один раз, а не пять.
con.execute("""
    WITH f AS (
        SELECT i.order_id, i.order_item_id, o.is_late
        FROM clean_items i
        JOIN clean_orders o USING (order_id)
        WHERE o.is_delivered
    )
    SELECT COUNT(*) FILTER (WHERE is_late)                  AS опоздавших_позиций,
           COUNT(DISTINCT order_id) FILTER (WHERE is_late)  AS опоздавших_заказов,
           ROUND(100.0 * COUNT(*) FILTER (WHERE is_late) /
                 NULLIF(COUNT(DISTINCT order_id) FILTER (WHERE is_late), 0) - 100, 1)
                                                            AS завышение_в_процентах
    FROM f
""").df()

## Собираем измерения

Измерения простые: берём справочники из CLEAN, добавляем читаемые атрибуты.
`dim_date` обычно генерируют, а не собирают
из фактов: иначе в календаре не будет дней без заказов, и отчёт «по дням»
незаметно пропустит пустые даты.

In [ ]:
con.execute("""
    CREATE OR REPLACE TABLE core.dim_date AS
    WITH bounds AS (
        SELECT MIN(CAST(order_purchase_timestamp AS DATE)) AS d0,
               MAX(CAST(order_purchase_timestamp AS DATE)) AS d1
        FROM clean_orders
    ),
    days AS (
        SELECT CAST(UNNEST(generate_series(d0, d1, INTERVAL 1 DAY)) AS DATE) AS full_date
        FROM bounds
    )
    SELECT CAST(strftime(full_date, '%Y%m%d') AS INTEGER) AS date_key,
           full_date,
           CAST(date_trunc('month', full_date) AS DATE)   AS month_start,
           EXTRACT(year  FROM full_date)                  AS year,
           EXTRACT(month FROM full_date)                  AS month,
           EXTRACT(dow   FROM full_date)                  AS day_of_week,
           EXTRACT(dow   FROM full_date) IN (0, 6)        AS is_weekend
    FROM days
""")

con.execute("""
    CREATE OR REPLACE TABLE core.dim_customer AS
    SELECT customer_id,
           customer_unique_id,
           customer_city,
           customer_state,
           customer_zip_code_prefix
    FROM clean_customers
""")

con.execute("""
    CREATE OR REPLACE TABLE core.dim_seller AS
    SELECT seller_id, seller_city, seller_state, seller_zip_code_prefix
    FROM clean_sellers
    UNION ALL
    -- «неизвестный участник» измерения: продавец, на которого ссылаются позиции,
    -- но которого нет в справочнике
    SELECT DISTINCT i.seller_id, 'unknown', 'XX', NULL
    FROM clean_items i
    WHERE NOT EXISTS (SELECT 1 FROM clean_sellers s WHERE s.seller_id = i.seller_id)
""")

con.execute(f"""
    CREATE OR REPLACE TABLE core.dim_product AS
    SELECT p.product_id,
           p.product_category_name,
           COALESCE(t.product_category_name_english, p.product_category_name)
               AS category_en,
           p.product_weight_g,
           p.product_photos_qty
    FROM clean_products p
    LEFT JOIN read_parquet(
        '{(paths.PARQUET / "product_category_name_translation.parquet").as_posix()}') t
      ON t.product_category_name = p.product_category_name
    UNION ALL
    -- «неизвестный участник» измерения: товар, на который ссылаются позиции
    -- заказа, но которого нет в справочнике товаров
    SELECT DISTINCT i.product_id, 'unknown', 'unknown',
           CAST(NULL AS DOUBLE), CAST(NULL AS DOUBLE)
    FROM clean_items i
    WHERE NOT EXISTS (SELECT 1 FROM clean_products p WHERE p.product_id = i.product_id)
""")

con.execute("""
    SELECT 'dim_date' AS измерение, COUNT(*) AS строк FROM core.dim_date
    UNION ALL SELECT 'dim_customer', COUNT(*) FROM core.dim_customer
    UNION ALL SELECT 'dim_seller', COUNT(*) FROM core.dim_seller
    UNION ALL SELECT 'dim_product', COUNT(*) FROM core.dim_product
""").df()

### Неизвестный участник измерения

В `dim_product` и `dim_seller` выше есть странная на первый взгляд часть:
мы добавляем в справочник строки-заглушки для товаров и продавцов, на которых
ссылаются позиции заказов, но которых в справочнике нет.

Здесь важно не смешать два понятия. **Late arriving dimension** – ситуация,
когда факт уже пришёл, а строки измерения ещё нет. **Unknown member** –
специально заведённая строка-заглушка. Она может временно помочь с late
arriving dimension, но это не синонимы: когда запись появится, ссылка на
заглушку должна быть сверена и заменена. Альтернативы хуже:

* выбросить такие позиции из факта – мы потеряем выручку, которая реально была;
* оставить ссылку «в никуда» – любой `JOIN` с измерением начнёт молча
  терять строки, и отчёты разойдутся.

Заглушка честнее: строка остаётся, деньги считаются, а в отчёте видно
`unknown` – и это сигнал разобраться с источником справочника.

## Что ломается в этом месте

**Справочник оказался не уникальным по своему ключу.**

In [ ]:
con.execute("""
    CREATE OR REPLACE TEMP TABLE dim_broken AS
    SELECT * FROM core.dim_seller
    UNION ALL
    SELECT * FROM core.dim_seller LIMIT 50            -- 50 продавцов задвоились
""")
res = con.execute("""
    SELECT (SELECT COUNT(*) FROM clean_items)                       AS позиций,
           (SELECT COUNT(*) FROM clean_items i JOIN core.dim_seller s USING (seller_id)) AS с_нормальным_измерением,
           (SELECT COUNT(*) FROM clean_items i JOIN dim_broken     s USING (seller_id)) AS со_сломанным
""").df()
print(res.to_string(index=False))
print("\nИзмерение обязано быть уникальным по своему ключу. Это не формальность:")
print("дубликат в справочнике молча размножает факты и завышает все суммы.")

**Посчитали среднее не на том уровне.**

In [ ]:
con.execute("""
    WITH f AS (
        SELECT i.order_id, o.delivery_days
        FROM clean_items i JOIN clean_orders o USING (order_id)
        WHERE o.is_delivered
    ),
    per_order AS (SELECT DISTINCT order_id, delivery_days FROM f)
    SELECT ROUND(AVG(delivery_days), 3) AS среднее_по_позициям FROM f
""").df().join(
    con.execute("""
        WITH f AS (
            SELECT DISTINCT i.order_id, o.delivery_days
            FROM clean_items i JOIN clean_orders o USING (order_id)
            WHERE o.is_delivered
        )
        SELECT ROUND(AVG(delivery_days), 3) AS среднее_по_заказам FROM f
    """).df())

Числа близкие, но разные – и разница систематическая: заказы с бо́льшим числом
позиций получают бо́льший вес. Пока это «третий знак», но на метрике, по которой
считают премию службе доставки, такие вещи замечают быстро.

### Задание 1. Таблица фактов `core.fact_order_items`

**Зерно: одна строка = одна позиция заказа.** Ключ – `(order_id, order_item_id)`.

| колонка | смысл |
|---------|-------|
| `order_id`, `order_item_id` | ключ факта |
| `date_key` | ключ измерения даты (`INTEGER`, вида `20180317`), по дате покупки |
| `order_date` | дата покупки (`DATE`) |
| `customer_id`, `product_id`, `seller_id` | ключи измерений |
| `price`, `freight_value` | меры уровня позиции |
| `order_status`, `is_delivered`, `is_late`, `delivery_days` | атрибуты уровня заказа |

Требования:

1. В факт попадают **только позиции тех заказов, которые есть в `clean_orders`**.
   Заказы, уехавшие в карантин в третьей работе, в модель не попадают: иначе нарушится
   ссылочная целостность.
2. Количество строк не должно отличаться от количества подходящих позиций
   в `clean_items`: соединение с заказом не имеет права ничего размножить.
3. Каждый `product_id`, `seller_id`, `customer_id`, `date_key` обязан
   находиться в соответствующем измерении.
4. `price` и `freight_value` не пусты.

Подумайте, какой тип соединения здесь уместен и почему `LEFT JOIN`
от `clean_items` был бы ошибкой.

In [ ]:
sql_fact = """
CREATE OR REPLACE TABLE core.fact_order_items AS
-- TODO: ваш запрос
SELECT 1
"""

con.execute(sql_fact)
con.execute("SELECT * FROM core.fact_order_items LIMIT 5").df()

In [ ]:
from checks.lab04 import (check_task_1, check_task_2, check_task_3, lab04_result)

check_task_1(con)

### Задание 2. Витрина `mart.mart_category_month`

Вопрос бизнеса: «как продаются категории товаров по месяцам и довольны ли
покупатели».

**Зерно: одна строка = (месяц, категория товара).**

| колонка | точное определение |
|---------|--------------------|
| `month` | первый день месяца покупки (`DATE`) |
| `category_en` | категория товара на английском из `core.dim_product` |
| `revenue` | сумма `price` по позициям этой категории в этом месяце |
| `n_items` | число позиций |
| `n_orders` | число **различных заказов**, в которых была позиция этой категории |
| `avg_review_score` | средняя оценка **по различным заказам** этой ячейки; заказ учитывается один раз независимо от числа позиций; если отзывов нет – `NULL` |

Источники: `core.fact_order_items`, `core.dim_product`, `clean_reviews`.

Главная ловушка здесь – `avg_review_score`. Отзыв относится к заказу,
а строки факта – к позициям. Если посчитать среднее «в лоб», заказ с пятью
позициями получит впятеро больший вес. Решите это осознанно.

In [ ]:
sql_mart = """
CREATE OR REPLACE TABLE mart.mart_category_month AS
-- TODO: ваш запрос
SELECT 1
"""

con.execute(sql_mart)
con.execute("SELECT * FROM mart.mart_category_month LIMIT 5").df()

In [ ]:
check_task_2(con)

### Задание 3. Пришло новое требование

Служба логистики просит:

> «Нам нужно понимать, как работают продавцы из разных штатов. Дайте по каждому
> месяцу и штату продавца: сколько доставленных заказов, какая доля из них
> опоздала и какой средний срок доставки. Заказ, в котором есть товары
> от продавцов из двух штатов, должен учитываться в обоих штатах –
> нам важна ответственность продавца, а не арифметическая чистота.»

Создайте `mart.mart_seller_delivery`:

**Зерно: одна строка = (месяц, штат продавца).**

| колонка | точное определение |
|---------|--------------------|
| `month` | первый день месяца покупки (`DATE`) |
| `seller_state` | штат продавца из `core.dim_seller` |
| `n_orders` | число **различных доставленных** заказов, в которых была позиция продавца из этого штата |
| `n_late_orders` | из них – число опоздавших (`is_late`) |
| `late_share` | `n_late_orders / n_orders` |
| `avg_delivery_days` | средний срок доставки **по заказам** (каждый заказ учитывается один раз) |

Учитываются только доставленные заказы (`is_delivered`).

Требование бизнеса прямо разрешает учитывать один заказ
в двух штатах. Сумма `n_orders` по всем штатам будет **больше** общего числа
доставленных заказов – и это не ошибка, а осознанное решение. Именно такие
вещи и надо фиксировать в описании витрины, иначе через месяц кто-нибудь
«исправит» её обратно.

In [ ]:
sql_mart2 = """
CREATE OR REPLACE TABLE mart.mart_seller_delivery AS
-- TODO: ваш запрос
SELECT 1
"""

con.execute(sql_mart2)
con.execute("SELECT * FROM mart.mart_seller_delivery LIMIT 5").df()

In [ ]:
check_task_3(con)

**Ваш ответ:** объясните в двух-трёх предложениях, почему сумма `n_orders`
по всем штатам больше общего числа доставленных заказов, и в каком случае
такую витрину нельзя показывать финансистам.

## Выгружаем витрины и смотрим на модель целиком

In [ ]:
for name in ["mart_category_month", "mart_seller_delivery"]:
    out = paths.MARTS / f"{name}.parquet"
    con.execute(f"COPY mart.{name} TO '{out.as_posix()}' (FORMAT PARQUET)")
    print(f"сохранено: {out.relative_to(paths.ROOT)}")

con.execute("""
    SELECT schema_name AS слой, table_name AS таблица, estimated_size AS строк
    FROM duckdb_tables()
    WHERE schema_name IN ('core', 'mart')
    ORDER BY 1, 2
""").df()

## Кто делает это в рабочих проектах

Всё, что мы сейчас написали руками, в индустрии обычно делают через **dbt**.
dbt не хранит данные и не считает их сам – он берёт ваши SQL-файлы, выстраивает
их в правильном порядке по зависимостям, создаёт таблицы/представления
в хранилище, прогоняет тесты и рисует граф происхождения данных.

Модель `mart_category_month` на dbt выглядела бы примерно так:

```sql
-- models/mart/mart_category_month.sql
{{ config(materialized='table') }}

with f as (
    select date_trunc('month', order_date)::date as month,
           p.category_en, f.order_id, f.price
    from {{ ref('fact_order_items') }} f
    join {{ ref('dim_product') }} p using (product_id)
)
select month, category_en, sum(price) as revenue, ...
from f group by 1, 2
```

```yaml
# models/mart/schema.yml
models:
  - name: mart_category_month
    columns:
      - name: month
        tests: [not_null]
      - name: revenue
        tests: [not_null]
```

Запись `ref()` даёт главное: dbt сам выстраивает порядок сборки – витрина
зависит от факта, факт от проверенного слоя, и перепутать их нельзя.
Это ровно та задача, которую мы будем решать руками в пятой работе.

**Когда dbt не нужен:** если у вас три таблицы и один скрипт. Инструмент
окупается, когда моделей десятки и над ними работают несколько человек.

## Та же задача в другой системе

Банк среднего размера. Есть:

* АБС (автоматизированная банковская система) с операциями по счетам – PostgreSQL;
* система заявок на кредиты;
* мобильное приложение, которое шлёт события (открыл экран, нажал кнопку);
* внешние данные бюро кредитных историй по запросу через API.

Три потребителя данных:

* **регуляторная отчётность** – раз в месяц, данные должны сходиться копейка
  в копейку и быть воспроизводимыми через три года;
* **риск-аналитика** – витрины по просрочкам, обновление раз в сутки;
* **скоринг при выдаче кредита** – ответ нужен за секунды в момент заявки.

Ответьте письменно:

1. Какое зерно вы выберете для основной таблицы фактов и почему?
2. Какие измерения понадобятся? Что будет меняться во времени и как вы
   это сохраните?
3. Одна витрина на всех или разные? Почему?
4. Где здесь достаточно пакетной обработки, а где нужен онлайн-доступ?
5. Почему регуляторную отчётность нельзя считать «поверх последней версии
   справочников»? Что для этого нужно в модели?
6. Нужен ли здесь Data Lake, если все данные и так табличные?
7. Какие компоненты были бы в этой задаче лишними?

### Ваш ответ

1. Зерно факта: …
2. Измерения и их история: …
3. Одна витрина или несколько: …
4. Пакетная обработка и онлайн: …
5. Воспроизводимость отчётности: …
6. Нужен ли Data Lake: …
7. Лишние компоненты: …

In [ ]:
lab04_result()
con.close()

## Вопросы для проверки себя

1. Чем слой CORE отличается от слоя CLEAN, если в обоих лежат «чистые данные»?
   Приведите пример изменения, которое затронет CORE, но не затронет CLEAN.
2. В таблице фактов с зерном «позиция заказа» есть колонка `delivery_days`.
   Какие агрегаты по ней корректны, а какие дадут смещённый результат?
3. Зачем нужен суррогатный ключ, если у нас уже есть уникальный `customer_id`?
   Назовите ситуацию, в которой без него будет больно.
4. Почему календарь `dim_date` генерируют, а не собирают из дат самих фактов?
5. Бизнес просит добавить в витрину новый разрез – «канал продаж».
   В каком слое вы будете вносить изменения и почему не в витрине?
6. Две витрины дают разные ответы на вопрос «сколько у нас заказов».
   Это всегда ошибка? Как отличить ошибку от разных, но законных определений?

## Что дальше

У нас есть модель с явным зерном: факты хранят события, измерения дают им
контекст, а витрина отвечает на конкретный вопрос бизнеса. Но пока это набор
ячеек, который кто-то обязан запускать в правильном порядке. В пятой работе
превратим её в обработку с зависимостями, логами и безопасным перезапуском.